In [1]:
import Pkg

In [2]:
Pkg.activate(".")
Pkg.add("ThreadPinning")
Pkg.add("KrylovKit")
Pkg.add("BenchmarkTools")
Pkg.add("ProfileCanvas")
Pkg.instantiate()

  Activating project at `~/Documents/KrylovKitBenchmark`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`


In [3]:
using Random
using KrylovKit: expintegrator, Arnoldi
using BenchmarkTools
using ProfileCanvas
using LinearAlgebra: norm

In [4]:
using ThreadPinning
pinthreads(:cores)
threadinfo()

Hostname: 	cuny
CPU(s): 	2 x Intel(R) Xeon(R) Gold 6226R CPU @ 2.90GHz
CPU target: 	cascadelake
Cores: 		32 (64 CPU-threads due to 2-way SMT)
NUMA domains: 	2 (16 cores each)

Julia threads: 	1

CPU socket 1
  0,32, 1,33, 2,34, 3,35, 4,36, 5,37, 6,38, 7,39, 
  8,40, 9,41, 10,42, 11,43, 12,44, 13,45, 14,46, 15,47

CPU socket 2
  16,48, 17,49, 18,50, 19,51, 20,52, 21,53, 22,54, 23,55, 
  24,56, 25,57, 26,58, 27,59, 28,60, 29,61, 30,62, 31,63


# = Julia thread, # = Julia thread on HT, # = >1 Julia thread

(Mapping: 1 => 0,)


In [5]:
@assert Threads.nthreads() == 1

In [6]:
filter(p -> contains(p[1], "THREAD"), ENV)

Dict{String, String} with 6 entries:
  "OPENBLAS_NUM_THREADS"   => "1"
  "VECLIB_MAXIMUM_THREADS" => "1"
  "OMP_NUM_THREADS"        => "1"
  "NUMEXPR_NUM_THREADS"    => "1"
  "MKL_NUM_THREADS"        => "1"
  "JULIA_NUM_THREADS"      => "1"

In [7]:
N = 100

100

In [8]:
"""Random complex matrix of dimension `N` with a spectral radius of approximately `ρ`."""
function random_matrix(N=N, ρ=1.0; rng=Random.GLOBAL_RNG)
    Δ = √(12 / N)
    X = Δ * (rand(rng, N, N) .- 0.5)
    Y = Δ * (rand(rng, N, N) .- 0.5)
    H = ρ * (X + Y * 1im) / √2
    return H
end

function random_hermitian_matrix(N=N, ρ=1.0; rng=Random.GLOBAL_RNG)
    Δ = √(12 / N)
    X = Δ * (rand(rng, N, N) .- 0.5)
    Y = Δ * (rand(rng, N, N) .- 0.5)
    Z = (X + Y * 1im) / √2
    H = ρ * (Z + Z') / (2 * √2)
    return H
end


"""Random normalized complex vector of dimension `N`"""
function random_state_vector(N=N; rng=Random.GLOBAL_RNG)
    Ψ = rand(rng, N) .* exp.((2π * im) .* rand(rng, N))
    Ψ ./= norm(Ψ)
    return Ψ
end

random_state_vector

In [9]:
struct Trajectory
    initial_state::Vector{ComplexF64}
    H::Matrix{ComplexF64}
    dt::Vector{Float64}
end

function Trajectory(;initial_state, H, nt)
    dt = rand(nt)
    N = length(initial_state)
    @assert size(H) == (N, N)
    Trajectory(initial_state, H, dt)
end

Trajectory

In [10]:
function propagate_traj(traj::Trajectory)
    # even if traj.H is Hermitian, we still use Arnoldi.
    # This propagation method is intended for non-Hermitian generators,
    # using a general matrix screws up the benchmark because the norm
    # of Ψ explodes.
    alg = Arnoldi()
    Ψ = traj.initial_state
    numops = 0
    for dt in traj.dt
        Ψ, info = expintegrator(traj.H, -1im * dt, (Ψ, ), alg)
        numops += info.numops
    end
    return numops
end

propagate_traj (generic function with 1 method)

In [11]:
traj100 = Trajectory(initial_state=random_state_vector(), H=random_hermitian_matrix(), nt=100);
propagate_traj(traj100)

3100

In [12]:
@benchmark propagate_traj(traj100)

BenchmarkTools.Trial: 155 samples with 1 evaluation.
 Range (min … max):  29.211 ms … 52.649 ms  ┊ GC (min … max): 0.00% … 26.70%
 Time  (median):     31.670 ms              ┊ GC (median):    3.41%
 Time  (mean ± σ):   32.266 ms ±  2.640 ms  ┊ GC (mean ± σ):  4.40% ±  3.31%

        ▃ ▆ ▃▄█ ▆▁ ▂                                           
  ▆▃▅▄▁██▇█▆████████▇▆▄▃▁▄▅▄▃▄▃▄▃▃▃▁▃▁▁▁▁▁▁▁▄▁▃▁▁▁▁▁▁▃▃▁▁▁▁▁▃ ▃
  29.2 ms         Histogram: frequency by time        40.6 ms <

 Memory estimate: 19.17 MiB, allocs estimate: 11200.

In [13]:
Base.GC.enable(false)
@benchmark propagate_traj(traj100)

BenchmarkTools.Trial: 129 samples with 1 evaluation.
 Range (min … max):  37.254 ms … 45.600 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     38.351 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   38.750 ms ±  1.471 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

   ▁ █▃ ▇ ▅▃▄                                                  
  ▆█▆██▇█████▇▄▇▃▃▃▃▁▄▁▄▁▃▅▃▃▁▄▁▃▃▅▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▃ ▃
  37.3 ms         Histogram: frequency by time        45.2 ms <

 Memory estimate: 19.17 MiB, allocs estimate: 11200.

In [14]:
Base.GC.enable(true)

false

In [15]:
@profview propagate_traj(traj100)

ProfileCanvas.ProfileData(Dict{String, ProfileCanvas.ProfileFrame}("1" => ProfileCanvas.ProfileFrame("root", "", "", 0, 54, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#15", "eventloop.jl", "/home/goerz/.julia/packages/IJulia/bHdNn/src/eventloop.jl", 38, 48, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("eventloop", "eventloop.jl", "/home/goerz/.julia/packages/IJulia/bHdNn/src/eventloop.jl", 8, 48, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("invokelatest", "essentials.jl", "./essentials.jl", 1052, 48, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#invokelatest#2", "essentials.jl", "./essentials.jl", 1055, 48, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("execute_request", "execute_request.jl", "/home/goerz/.julia/packages/IJulia/bHdNn/src/execute_request.jl", 67, 48, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("softscope_include_string", "SoftGlobalScope.jl", "/home/goerz/.julia/packages/SoftGlobalScope/u4UzH/src/SoftGlobalScope.jl", 65, 48, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("include_string", "loading.jl", "./loading.jl", 2734, 48, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("eval", "boot.jl", "./boot.jl", 430, 48, missing, 0x01, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("propagate_traj", "In[10]", "./In[10]", 10, 45, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("expintegrator", "expintegrator.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/matrixfun/expintegrator.jl", 106, 45, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("expintegrator", "expintegrator.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/matrixfun/expintegrator.jl", 267, 17, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("expand!", "arnoldi.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/factorizations/arnoldi.jl", 196, 17, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#expand!#31", "arnoldi.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/factorizations/arnoldi.jl", 206, 16, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("arnoldirecurrence!", "arnoldi.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/factorizations/arnoldi.jl", 236, 11, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("orthogonalize!", "orthonormal.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/orthonormal.jl", 474, 9, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("orthogonalize!", "orthonormal.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/orthonormal.jl", 449, 7, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("dot", "matmul.jl", "/cache/build/builder-amdci5-5/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/matmul.jl", 19, 7, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("dotc", "blas.jl", "/cache/build/builder-amdci5-5/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/blas.jl", 398, 7, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("dotc", "blas.jl", "/cache/build/builder-amdci5-5/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/blas.jl", 365, 7, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])])]), ProfileCanvas.ProfileFrame("orthogonalize!", "orthonormal.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/orthonormal.jl", 450, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("axpy!", "generic.jl", "/cache/build/builder-amdci5-5/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/generic.j

In [16]:
traj1000 = Trajectory(initial_state=random_state_vector(), H=random_hermitian_matrix(), nt=1000);
propagate_traj(traj1000)

31000

In [17]:
@benchmark propagate_traj($traj1000)

BenchmarkTools.Trial: 16 samples with 1 evaluation.
 Range (min … max):  309.377 ms … 346.817 ms  ┊ GC (min … max): 1.02% … 2.71%
 Time  (median):     313.384 ms               ┊ GC (median):    4.59%
 Time  (mean ± σ):   317.117 ms ±   9.467 ms  ┊ GC (mean ± σ):  4.06% ± 1.16%

  ▁ █▁█▁▁▁█      ▁▁         ▁ ▁                               ▁  
  █▁███████▁▁▁▁▁▁██▁▁▁▁▁▁▁▁▁█▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  309 ms           Histogram: frequency by time          347 ms <

 Memory estimate: 191.70 MiB, allocs estimate: 112000.

In [18]:
@profview propagate_traj(traj1000)

ProfileCanvas.ProfileData(Dict{String, ProfileCanvas.ProfileFrame}("1" => ProfileCanvas.ProfileFrame("root", "", "", 0, 331, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#15", "eventloop.jl", "/home/goerz/.julia/packages/IJulia/bHdNn/src/eventloop.jl", 38, 317, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("eventloop", "eventloop.jl", "/home/goerz/.julia/packages/IJulia/bHdNn/src/eventloop.jl", 8, 317, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("invokelatest", "essentials.jl", "./essentials.jl", 1052, 317, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#invokelatest#2", "essentials.jl", "./essentials.jl", 1055, 317, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("execute_request", "execute_request.jl", "/home/goerz/.julia/packages/IJulia/bHdNn/src/execute_request.jl", 67, 317, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("softscope_include_string", "SoftGlobalScope.jl", "/home/goerz/.julia/packages/SoftGlobalScope/u4UzH/src/SoftGlobalScope.jl", 65, 317, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("include_string", "loading.jl", "./loading.jl", 2734, 317, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("eval", "boot.jl", "./boot.jl", 430, 317, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("propagate_traj", "In[10]", "./In[10]", 10, 317, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("expintegrator", "expintegrator.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/matrixfun/expintegrator.jl", 106, 317, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("expintegrator", "expintegrator.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/matrixfun/expintegrator.jl", 267, 186, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("expand!", "arnoldi.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/factorizations/arnoldi.jl", 196, 186, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#expand!#31", "arnoldi.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/factorizations/arnoldi.jl", 206, 183, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("arnoldirecurrence!", "arnoldi.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/factorizations/arnoldi.jl", 236, 104, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("orthogonalize!", "orthonormal.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/orthonormal.jl", 475, 53, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("reorthogonalize!", "orthonormal.jl", "/home/goerz/.julia/packages/KrylovKit/diNbc/src/orthonormal.jl", 463, 32, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("axpy!", "generic.jl", "/cache/build/builder-amdci5-5/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/generic.jl", 1523, 32, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("axpy!", "blas.jl", "/cache/build/builder-amdci5-5/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/blas.jl", 525, 32, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("axpy!", "blas.jl", "/cache/build/builder-amdci5-5/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/blas.jl", 513, 28, missing, 0x00, missing, ProfileCanvas.ProfileFrame[]), ProfileCanvas.ProfileFrame("axpy!", "blas.jl", "/cache/build/builder-amdci5-5/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/blas.jl", 512, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])])]), ProfileCanvas.ProfileFrame("reorthogonalize!", "orthonormal.jl", "/home/goerz/.julia/packages/Kry